# 1. Exploratory Data Analysis (EDA)

This notebook performs EDA on the downsampled BigEarthNet-S2 dataset (25k patches, single-label).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import rasterio

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

META_PATH = 'data/processed/bigearthnet_25k_singlelabel/metadata.json'
with open(META_PATH, 'r') as f:
    data = json.load(f)
patches = data['patches']
stats = data['band_stats']
print(f'Total patches: {len(patches)}')
print(f'Unique labels: {len(set(p["single_label"] for p in patches))}')

## Class Distribution

In [ ]:
labels = [p['single_label'] for p in patches]
label_counts = pd.Series(labels).value_counts()
label_counts.plot(kind='bar')
plt.title('Class Distribution (Single-Label)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## Band Statistics

In [ ]:
mean = np.array(stats['mean'])
std = np.array(stats['std'])
band_names = [f'B{i}' for i in range(1, 11)]
x = np.arange(len(band_names))
plt.bar(x, mean, yerr=std, capsize=4)
plt.xticks(x, band_names)
plt.title('Per-Band Mean and Std')
plt.ylabel('Pixel Intensity')
plt.show()

## Sample RGB Composite

In [ ]:
def read_bands(patch_dir, target_size=120):
    tifs = sorted(Path(patch_dir).glob('*.tif'))
    bands = []
    for tif in tifs:
        with rasterio.open(tif) as src:
            band = src.read(1).astype(np.float32)
            if band.shape != (target_size, target_size):
                import torch
                band = torch.tensor(band).unsqueeze(0).unsqueeze(0)
                band = torch.nn.functional.interpolate(band, size=(target_size, target_size), mode='bilinear', align_corners=False)
                band = band.squeeze().numpy()
            bands.append(band)
    return np.stack(bands, axis=0)

# Pick 3 random patches
sample_patches = np.random.choice(patches, 3, replace=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, sample_patches):
    img = read_bands(p['patch_dir'])
    # Simple RGB: use bands 4,3,2 (B04=red, B03=green, B02=blue) if available, else first 3
    rgb = np.stack([img[3], img[2], img[1]], axis=-1)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
    ax.imshow(rgb)
    ax.set_title(p['single_label'])
    ax.axis('off')
plt.show()